## Connexion à PostgreSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================
# ENV SETUP
# =========================
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

encoded_password = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{encoded_password}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)


## Création des tables

In [ ]:
from sqlalchemy import create_engine, text

sql = """
-- Table des agences
CREATE TABLE IF NOT EXISTS agences (
    agence_id SERIAL PRIMARY KEY,
    agence VARCHAR(100) UNIQUE
);

-- Table des segments clients
CREATE TABLE IF NOT EXISTS segments_client (
    segment_id SERIAL PRIMARY KEY,
    segment_client VARCHAR(50) UNIQUE,
    Categorie_risque VARCHAR(50)
);

-- Table des clients
CREATE TABLE IF NOT EXISTS clients (
    client_id VARCHAR PRIMARY KEY,
    segment_client VARCHAR(50),
    score_credit_client NUMERIC(10,2)
);

-- Table des produits
CREATE TABLE IF NOT EXISTS produits (
    produit_id SERIAL PRIMARY KEY,
    produit VARCHAR(100) UNIQUE
);

-- Table des transactions
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id BIGINT PRIMARY KEY,
    client_id VARCHAR,
    date_transaction DATE,
    montant NUMERIC(15,2),
    devise VARCHAR(10),
    taux_change_eur NUMERIC(10,4),
    montant_eur NUMERIC(15,2),
    categorie VARCHAR(100),
    produit VARCHAR(100),
    agence VARCHAR(100),
    type_operation VARCHAR(50),
    statut VARCHAR(50),
    score_credit_client NUMERIC(10,2),
    segment_client VARCHAR(50),
    solde_avant NUMERIC(15,2),
    anomaly_montant NUMERIC(10,2),
    anomaly_score NUMERIC(10,2),
    is_anomaly BOOLEAN,
    "Années" INT,
    "Mois" INT,
    "Trimestre" INT,
    "jour-semaine" VARCHAR(20),
    montant_eur_verifie NUMERIC(15,2),
    Comparaison VARCHAR(50),
    Categorie_risque VARCHAR(50),
    taux_rejet NUMERIC(10,2),
    CONSTRAINT fk_transactions_clients
        FOREIGN KEY (client_id) REFERENCES clients(client_id)
);
"""

with engine.begin() as connection:
    connection.execute(text(sql))

print("✅ Tables créées avec succès")

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE INDEX idx_fact_client ON fact_transactions(client_id);
        CREATE INDEX idx_fact_date ON fact_transactions(date_id);
        CREATE INDEX idx_fact_agence ON fact_transactions(agence_id);
'''))

In [ ]:
import pandas as pd

df = pd.read_csv('financecore_clean.csv')
df

In [ ]:
df.columns

In [ ]:
agences_df = df["agence"].drop_duplicates()
segments_clients_df = df[["segment_client", "Categorie_risque"]].drop_duplicates()
clients_df = df[["client_id", "segment_client", "score_credit_client"]].drop_duplicates()
produits_df = df[["produit"]].drop_duplicates()
transactions_df = df[[
        "transaction_id",
        "client_id",
        "date_transaction",
        "montant",
        "devise",
        "taux_change_eur",
        "montant_eur",
        "categorie",
        "produit",
        "agence",
        "type_operation",
        "statut",
        "score_credit_client",
        "segment_client",
        "solde_avant",
        "anomaly_montant",
        "anomaly_score",
        "is_anomaly",
        "Années",
        "Mois",
        "Trimestre",
        "jour-semaine",
        "montant_eur_verifie",
        "Comparaison",
        "Categorie_risque",
        "taux_rejet"
    ]].drop_duplicates(subset="transaction_id")

In [ ]:
agences_df.to_sql("agences", engine, if_exists="replace", index=False)
segments_clients_df.to_sql("segments_client", engine, if_exists="replace", index=False)
produits_df.to_sql("produits", engine, if_exists="replace", index=False)
transactions_df.to_sql("transactions", engine, if_exists="replace", index=False)

print("✅ Données insérées avec succès dans toutes les tables")

In [ ]:
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS clients CASCADE"))
clients_df.to_sql("clients", engine, if_exists="replace", index=False)